In [1]:
#import pandas as pd
#import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
import optuna

from catboost import CatBoostClassifier
from catboost import Pool

import gc

In [2]:
df_train = pd.read_csv('data/train.csv', index_col='id')
df_test = pd.read_csv('data/test.csv', index_col='id')
df_train.shape, df_test.shape

((11504798, 11), (7669866, 10))

In [3]:
# change column names to be all lowercase
df_train.rename(columns=lambda col : col.lower(), inplace=True)
df_test.rename(columns=lambda col : col.lower(), inplace=True)

df_train.head(10)

,gender,age,driving_license,region_code,previously_insured,vehicle_age,vehicle_damage,annual_premium,policy_sales_channel,vintage,response
id,,,,,,,,,,,
0,Male,21,1,35.0,0,1-2 Year,Yes,65101.0,124.0,187,0
1,Male,43,1,28.0,0,> 2 Years,Yes,58911.0,26.0,288,1
2,Female,25,1,14.0,1,< 1 Year,No,38043.0,152.0,254,0
3,Female,35,1,1.0,0,1-2 Year,Yes,2630.0,156.0,76,0
4,Female,36,1,15.0,1,1-2 Year,No,31951.0,152.0,294,0
5,Female,31,1,47.0,1,< 1 Year,No,28150.0,152.0,197,0
6,Male,23,1,45.0,1,< 1 Year,No,27128.0,152.0,190,0
7,Female,47,1,8.0,0,1-2 Year,Yes,40659.0,26.0,262,1
8,Female,26,1,28.0,1,< 1 Year,No,31639.0,152.0,36,0


In [4]:
drop_idx = df_train.loc[df_train['region_code'] == 39.2].index[0]
df_train.drop(index=drop_idx, inplace=True)
df_train.reset_index(drop=True, inplace=True)

df_train.shape

(11504797, 11)

### Handling categorical features in Catboost
- with catboost, we can specify which columns in our input dataframe/array are categorical
    - if using native API, we can specify cat cols when putting the data into a Pool object
    - with sklearn API (e.g. CatBoostClassifier), model object's .fit() method has a cat_features parameter we can set
- so we can just encode categorical features to have integer values first, before feeding it to the model
- NOTE: for continuous features, catboost quantizes (i.e. puts into finite number of buckets/bins) them automatically

In [5]:
def simple_preprocessing(df, is_train=False):
    df = df.copy()

    df['age'] = df['age'].astype('int8')
    df['driving_license'] = df['driving_license'].astype('int8')
    df['region_code'] = df['region_code'].astype('int8')
    df['previously_insured'] = df['previously_insured'].astype('int8')
    df['annual_premium'] = df['annual_premium'].astype('int32')
    df['policy_sales_channel'] = df['policy_sales_channel'].astype('int16')
    df['vintage'] = df['vintage'].astype('int16')
    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1}).astype('int8')
    df['vehicle_age'] = df['vehicle_age'].map({'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 3}).astype('int8')
    df['vehicle_damage'] = df['vehicle_damage'].map({'No': 0, 'Yes': 1}).astype('int8')

    if is_train==True:

        df['response'] = df['response'].astype('int8')

    return df

train = simple_preprocessing(df_train, True)
test = simple_preprocessing(df_test, False)

In [6]:
train.head(10)

,gender,age,driving_license,region_code,previously_insured,vehicle_age,vehicle_damage,annual_premium,policy_sales_channel,vintage,response
0,1,21,1,35,0,1,1,65101,124,187,0
1,1,43,1,28,0,3,1,58911,26,288,1
2,0,25,1,14,1,0,0,38043,152,254,0
3,0,35,1,1,0,1,1,2630,156,76,0
4,0,36,1,15,1,1,0,31951,152,294,0
5,0,31,1,47,1,0,0,28150,152,197,0
6,1,23,1,45,1,0,0,27128,152,190,0
7,0,47,1,8,0,1,1,40659,26,262,1
8,0,26,1,28,1,0,0,31639,152,36,0
9,0,66,1,11,0,1,1,2630,26,125,0


In [7]:
TARGET = 'response'

features = list(df_test.columns)
#features = ['gender', 'driving_license', 'previously_insured', 'vehicle_age', 'vehicle_damage', 'policy_sales_channel', 'region_code', 'age', 'vintage', 'annual_premium']

### Hyperparameter optimization

In [ ]:
X = train.drop(columns=['response'])
y = train['response'].values

In [ ]:
def objective(trial):
    X_sub, _, y_sub, _ = train_test_split(X, y, test_size=0.6, shuffle=True, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_sub, y_sub, test_size=0.2, shuffle=True, stratify=y_sub)

    params = {
        "objective": "Logloss",
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.01, 1, log=True),
        "depth": trial.suggest_int("depth", 1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.5),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 2.0),
        "eval_metric": "AUC",
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0., 10.),
        "iterations": 1500,
        "fold_permutation_block": trial.suggest_int("fold_permutation_block", 30, 60)
    }
    
    cb_model = CatBoostClassifier(**params)
    
    cb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        cat_features=np.array(features),
        verbose=0,
        early_stopping_rounds=100
    )

    val_pred = cb_model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val, val_pred)

    return score

In [ ]:
# NOTE: optimization is slow without GPU availablity

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

### Catboost training, performance evaluation

In [ ]:
k=5
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# object for storing OOF predictions on training data
val_preds = np.zeros(shape=(train.shape[0], 2))
# object for storing test data predictions for each split's catboost model instance
test_preds = np.zeros(shape=(test.shape[0], k))

for i, (train_idx, val_idx) in enumerate(cv.split(train, train[TARGET])):
    print("########## Fold {} ########## \n".format(i))
    X_train = train.loc[train_idx, features]
    y_train = train.loc[train_idx, TARGET].values
    X_val = train.loc[val_idx, features]
    y_val = train.loc[val_idx, TARGET].values
    
    train_pool = Pool(X_train, y_train, cat_features=features)
    val_pool = Pool(X_val, y_val, cat_features=features)
    test_pool = Pool(test, cat_features=features)
    
    params = {
    'loss_function': 'Logloss',
    'iterations':5000,
    'learning_rate':0.01,
    'depth':8,
    'l2_leaf_reg':0.1,
    'verbose':False,
    'eval_metric':'AUC',
    'random_seed':42,
    'task_type':'GPU'
    }

    model = CatBoostClassifier(**params)
    
    # NOTE: model training is slow without GPU access!
    model.fit(
        train_pool, 
        eval_set=val_pool, 
        early_stopping_rounds=200,
        verbose=500
    )
    
    fold_roc = model.best_score_['validation']['AUC']
    print("fold AUC: {:.4f}".format(fold_roc))
    
    val_preds[val_idx] = model.predict_proba(val_pool)[:, 1]
    test_preds[:, i] = model.predict_proba(test)[:,1]
 
overall_val_score = roc_auc_score(train[TARGET], val_preds)
print("Overall validation AUC score: {:.4f}".format(overall_val_score))

In [ ]:
# the mean predictions from across all k-fold model instances can serve as final catboost predictions
catboost_predictions = np.mean(test_preds, axis=1)

catboost_results = pd.Series(data = catboost_predictions, index=test.index, name="Response")
filename = "catboost_results.csv"
catboost_results.to_csv(filename)